# SynthRAD2025 Multi-task CycleGAN — Results Analysis

This notebook loads ablation training logs and test-set CSV files produced by
`evaluate/test_multitask.py`, then generates all figures used in the paper.

**Run order:** execute cells top-to-bottom after setting `ANATOMY` and `OUTPUT_DIR`.

## Cell 1 — Imports and configuration

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

# ── User-configurable variables ──────────────────────────────────────────────
ANATOMY      = 'head_neck'          # 'head_neck' | 'thorax' | 'abdomen'
OUTPUT_DIR   = Path('..')           # project root (relative to notebooks/)

ABLATION_NAMES: List[str] = [
    'baseline_cyclegan',
    'paired_cyclegan',
    'plus_seg_loss',
    'plus_anatomy_consistency',
    'no_perceptual',
    'no_warmup',
]

ORGAN_NAMES_BY_ANATOMY: Dict[str, List[str]] = {
    'head_neck': ['brainstem', 'parotid_L', 'parotid_R', 'mandible', 'spinal_cord'],
    'thorax':    ['lung_L', 'lung_R', 'heart', 'spinal_cord', 'esophagus'],
    'abdomen':   ['liver', 'kidney_L', 'kidney_R', 'spleen', 'pancreas'],
}

FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ORGAN_NAMES = ORGAN_NAMES_BY_ANATOMY[ANATOMY]
print(f'Anatomy : {ANATOMY}')
print(f'Organs  : {ORGAN_NAMES}')
print(f'Figures will be saved to: {FIGURES_DIR.resolve()}')

## Cell 2 — Load ablation CSVs

In [ ]:
def _load_training_csv(ablation: str, anatomy: str, output_dir: Path) -> Optional[pd.DataFrame]:
    """Load the per-epoch validation CSV written by train_multitask.py."""
    csv_path = output_dir / 'logs' / f'{ablation}_{anatomy}.csv'
    if not csv_path.exists():
        print(f'  [missing] {csv_path}')
        return None
    df = pd.read_csv(csv_path)
    df['ablation'] = ablation
    return df

def _load_test_csv(ablation: str, anatomy: str, output_dir: Path) -> Optional[pd.DataFrame]:
    """Load the per-patient test results CSV written by test_multitask.py."""
    csv_path = output_dir / 'results' / f'{ablation}_{anatomy}_test_results.csv'
    if not csv_path.exists():
        print(f'  [missing] {csv_path}')
        return None
    df = pd.read_csv(csv_path)
    df['ablation'] = ablation
    return df

print('Loading training-curve CSVs ...')
train_dfs: Dict[str, pd.DataFrame] = {}
for abl in ABLATION_NAMES:
    df = _load_training_csv(abl, ANATOMY, OUTPUT_DIR)
    if df is not None:
        train_dfs[abl] = df
        print(f'  {abl:30s}  {len(df)} epochs')

print('\nLoading test-result CSVs ...')
test_dfs: Dict[str, pd.DataFrame] = {}
for abl in ABLATION_NAMES:
    df = _load_test_csv(abl, ANATOMY, OUTPUT_DIR)
    if df is not None:
        test_dfs[abl] = df
        print(f'  {abl:30s}  {len(df)} patients')

# Concatenated views (for seaborn plots)
all_train = pd.concat(train_dfs.values(), ignore_index=True) if train_dfs else pd.DataFrame()
all_test  = pd.concat(test_dfs.values(),  ignore_index=True) if test_dfs  else pd.DataFrame()

print(f'\nTotal train rows: {len(all_train)}')
print(f'Total test rows : {len(all_test)}')

## Cell 3 — Training curves

In [ ]:
if not train_dfs:
    print('No training CSVs found — skipping training-curve plot.')
else:
    n_abl  = len(train_dfs)
    n_cols = min(n_abl, 3)
    n_rows = (n_abl + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        squeeze=False,
    )
    fig.suptitle(f'Training curves — {ANATOMY.upper()}', fontsize=14, y=1.02)

    for idx, (abl, df) in enumerate(train_dfs.items()):
        ax  = axes[idx // n_cols][idx % n_cols]
        ax2 = ax.twinx()

        epochs = df['epoch'] if 'epoch' in df.columns else range(len(df))

        # Primary axis — SSIM
        if 'mr2ct_ssim' in df.columns:
            ax.plot(epochs, df['mr2ct_ssim'], color='steelblue', lw=1.8,
                    label='MR→CT SSIM')
        if 'ct2mr_ssim' in df.columns:
            ax.plot(epochs, df['ct2mr_ssim'], color='darkorange', lw=1.8,
                    linestyle='--', label='CT→MR SSIM')
        ax.set_ylabel('SSIM', color='steelblue')
        ax.set_ylim(0, 1)
        ax.tick_params(axis='y', labelcolor='steelblue')

        # Secondary axis — losses
        for loss_key, colour, ls in [
            ('loss_seg',      'forestgreen', '-'),
            ('loss_anatomy',  'crimson',     '--'),
        ]:
            if loss_key in df.columns:
                ax2.plot(epochs, df[loss_key], color=colour, lw=1.0,
                         linestyle=ls, alpha=0.6, label=loss_key)
        ax2.set_ylabel('Loss', color='grey')
        ax2.tick_params(axis='y', labelcolor='grey')

        ax.set_title(abl, fontsize=10)
        ax.set_xlabel('Epoch')

        # Combined legend
        lines1, labs1 = ax.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc='lower right')

    # Hide unused subplot panels
    for idx in range(n_abl, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    plt.tight_layout()
    fig_path = FIGURES_DIR / f'{ANATOMY}_training_curves.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

## Cell 4 — Ablation comparison bar chart

In [ ]:
if all_test.empty:
    print('No test CSVs found — skipping bar chart.')
else:
    metrics_to_plot = ['mr2ct_ssim', 'ct2mr_ssim']
    metric_labels   = {'mr2ct_ssim': 'MRI→CT SSIM', 'ct2mr_ssim': 'CT→MRI SSIM'}

    # Build tidy dataframe for seaborn
    tidy_rows = []
    for abl in ABLATION_NAMES:
        if abl not in test_dfs:
            continue
        df = test_dfs[abl]
        for metric in metrics_to_plot:
            if metric not in df.columns:
                continue
            for val in df[metric].dropna():
                tidy_rows.append({'ablation': abl, 'metric': metric_labels[metric],
                                  'value': val})
    tidy = pd.DataFrame(tidy_rows)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(
        data=tidy, x='ablation', y='value', hue='metric',
        palette=['steelblue', 'darkorange'],
        capsize=0.06, errwidth=1.5,
        ax=ax,
    )
    ax.set_title(f'Ablation SSIM comparison — {ANATOMY.upper()}', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('SSIM')
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.legend(title='Direction')

    plt.tight_layout()
    fig_path = FIGURES_DIR / f'{ANATOMY}_ablation_bar.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

## Cell 5 — Dice per organ class heatmap

In [ ]:
if all_test.empty:
    print('No test CSVs found — skipping Dice heatmap.')
else:
    # Dice class columns are dice_class_1 … dice_class_5
    n_fg = len(ORGAN_NAMES)  # foreground classes (background excluded)
    dice_cols = [f'dice_class_{i}' for i in range(1, n_fg + 1)]

    heat_data = []
    row_labels = []
    for abl in ABLATION_NAMES:
        if abl not in test_dfs:
            continue
        df = test_dfs[abl]
        means = [df[c].mean() if c in df.columns else np.nan for c in dice_cols]
        heat_data.append(means)
        row_labels.append(abl)

    if heat_data:
        heat_df = pd.DataFrame(
            heat_data,
            index=row_labels,
            columns=ORGAN_NAMES,
        )

        fig, ax = plt.subplots(figsize=(max(8, n_fg * 1.5), len(row_labels) * 0.8 + 1.5))
        sns.heatmap(
            heat_df,
            ax=ax,
            vmin=0, vmax=1,
            cmap='YlOrRd',
            annot=True, fmt='.3f',
            linewidths=0.5,
            cbar_kws={'label': 'Dice'},
        )
        ax.set_title(f'Mean Dice per organ — {ANATOMY.upper()}', fontsize=13)
        ax.set_xlabel('Organ')
        ax.set_ylabel('Ablation')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

        plt.tight_layout()
        fig_path = FIGURES_DIR / f'{ANATOMY}_dice_heatmap.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fig_path}')
    else:
        print('No Dice data available for heatmap.')

## Cell 6 — Qualitative results display

In [ ]:
def _load_midslice(nifti_path: Path) -> Optional[np.ndarray]:
    """Load the middle axial slice from a NIfTI volume."""
    try:
        import SimpleITK as sitk
        img = sitk.ReadImage(str(nifti_path))
        arr = sitk.GetArrayFromImage(img)  # (Z, Y, X)
        mid = arr.shape[0] // 2
        return arr[mid].astype(np.float32)
    except Exception as exc:
        print(f'  Could not load {nifti_path}: {exc}')
        return None

def _norm(arr: np.ndarray) -> np.ndarray:
    """Normalise to [0, 1] for display."""
    lo, hi = arr.min(), arr.max()
    if hi == lo:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# Full-model ablation and first available test patient
QUALITATIVE_ABLATION = 'plus_anatomy_consistency'

split_path = OUTPUT_DIR / 'splits' / f'{ANATOMY}_split.json'
patient_id = None
if split_path.exists():
    with open(split_path) as fh:
        split = json.load(fh)
    test_list = split.get('test', [])
    if test_list:
        patient_id = test_list[0]['patient_id']

if patient_id is None:
    print('No test patient found — skipping qualitative display.')
else:
    patient_out = OUTPUT_DIR / 'outputs' / ANATOMY / patient_id

    # Load real MRI (from split entry), synthetic CT, real CT, seg pred
    real_mr_path   = Path(test_list[0]['mr_path'])  if test_list else None
    real_ct_path   = Path(test_list[0]['ct_path'])  if test_list else None
    fake_ct_path   = patient_out / 'synthetic_ct.nii.gz'
    seg_pred_path  = patient_out / 'seg_pred.nii.gz'

    slices = {
        'Real MRI':        _load_midslice(real_mr_path)  if real_mr_path  else None,
        'Synthetic CT':    _load_midslice(fake_ct_path),
        'Real CT':         _load_midslice(real_ct_path)  if real_ct_path  else None,
        'Seg Prediction':  _load_midslice(seg_pred_path),
    }

    available = {k: v for k, v in slices.items() if v is not None}

    if len(available) < 2:
        print('Insufficient NIfTI files found — skipping qualitative display.')
    else:
        n_panels = len(available)
        fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
        if n_panels == 1:
            axes = [axes]

        panel_titles = list(available.keys())

        for ax, title in zip(axes, panel_titles):
            sl = available[title]
            if title == 'Seg Prediction':
                n_cls = len(ORGAN_NAMES) + 1  # includes background
                ax.imshow(sl, cmap='tab10', vmin=0, vmax=n_cls - 1, interpolation='nearest')
            elif title == 'Synthetic CT' and 'Seg Prediction' in available:
                # Synthesis + segmentation overlay
                ax.imshow(_norm(sl), cmap='gray')
                seg_sl = available['Seg Prediction']
                masked_seg = np.ma.masked_where(seg_sl == 0, seg_sl)
                ax.imshow(masked_seg, cmap='tab10', alpha=0.4,
                          vmin=1, vmax=len(ORGAN_NAMES))
            else:
                ax.imshow(_norm(sl), cmap='gray')

            # Difference map for synthetic vs real CT
            if (title == 'Synthetic CT'
                    and 'Real CT' in available
                    and available['Real CT'] is not None):
                diff = available['Synthetic CT'] - available['Real CT']
                # Add an inset difference map
                ax_ins = ax.inset_axes([0.0, 0.0, 0.35, 0.35])
                ax_ins.imshow(diff, cmap='bwr',
                              vmin=-np.abs(diff).max(),
                              vmax= np.abs(diff).max())
                ax_ins.set_title('Δ CT', fontsize=7)
                ax_ins.axis('off')

            ax.set_title(title, fontsize=11)
            ax.axis('off')

        fig.suptitle(f'Patient: {patient_id} | {ANATOMY.upper()}', fontsize=12)
        plt.tight_layout()
        fig_path = FIGURES_DIR / f'{ANATOMY}_qualitative_{patient_id}.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fig_path}')

## Cell 7 — Statistical significance testing

In [ ]:
FULL_MODEL   = 'plus_anatomy_consistency'
BASELINE_ABL = 'paired_cyclegan'
ALPHA        = 0.05

if FULL_MODEL not in test_dfs or BASELINE_ABL not in test_dfs:
    print(f'Cannot run t-test: need both {FULL_MODEL!r} and {BASELINE_ABL!r} test CSVs.')
else:
    df_full     = test_dfs[FULL_MODEL]
    df_baseline = test_dfs[BASELINE_ABL]

    # Align on patient_id to ensure paired comparison
    common_ids = set(df_full['patient_id']) & set(df_baseline['patient_id'])
    if len(common_ids) < 2:
        print(f'Only {len(common_ids)} common patients — cannot compute paired t-test.')
    else:
        df_full_a = df_full[df_full['patient_id'].isin(common_ids)].sort_values('patient_id')
        df_base_a = df_baseline[df_baseline['patient_id'].isin(common_ids)].sort_values('patient_id')

        print(f'Paired t-test: {FULL_MODEL!r} vs {BASELINE_ABL!r}')
        print(f'n = {len(common_ids)} patients  |  α = {ALPHA}\n')
        print(f'{"Metric":<20} {"Full model":>12} {"Baseline":>12} {"t-stat":>10} {"p-value":>12} {"Significant":>12}')
        print('-' * 82)

        for metric, label in [
            ('mr2ct_ssim', 'MR→CT SSIM'),
            ('mean_dice',  'Mean Dice'),
        ]:
            if metric not in df_full_a.columns or metric not in df_base_a.columns:
                print(f'{label:<20}  [column not found]')
                continue

            a = df_full_a[metric].values
            b = df_base_a[metric].values

            t_stat, p_val = stats.ttest_rel(a, b, alternative='two-sided')

            sig_str = 'YES *' if p_val < ALPHA else 'no'
            if p_val < 0.001:
                sig_str = 'YES ***'
            elif p_val < 0.01:
                sig_str = 'YES **'

            print(
                f'{label:<20}'
                f'{a.mean():>12.4f}'
                f'{b.mean():>12.4f}'
                f'{t_stat:>10.3f}'
                f'{p_val:>12.4f}'
                f'{sig_str:>12}'
            )

        print()
        print(f'Interpretation:')
        print(f'  * p < {ALPHA}  → statistically significant difference at α={ALPHA}')
        print(f'  ** p < 0.01, *** p < 0.001')
        print(f'  Positive t-stat → full model outperforms {BASELINE_ABL!r} on that metric.')
        print(f'  scipy.stats.ttest_rel — paired, two-tailed.')